<a href="https://colab.research.google.com/github/niniqoiii/Springfield/blob/main/inference.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Simpsons Inference Notebook

This notebook loads `model.pth` and `classes.json`, downloads/extracts the Simpsons ZIP from Google Drive, prepares a test image folder, and creates `results.json`.

**Important:** Upload `model.pth` and `classes.json` to Colab before running all cells.

In [13]:
!pip install -q gdown

import os
import json
import zipfile
import shutil
import gdown
from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import transforms

print('Imports ready')

Imports ready


In [14]:
# Download and extract dataset ZIP from Google Drive
file_id = '12FqLnmk6Cu0NR8UehRQiD7az2iDYufCX'
zip_file = '/content/Simpsons.zip'
extract_dir = '/content/simpsons_extracted'

if not os.path.exists(zip_file):
    print('Downloading Simpsons.zip...')
    gdown.download(f'https://drive.google.com/uc?id={file_id}', zip_file, quiet=False)
else:
    print('ZIP already exists')

if os.path.exists(extract_dir):
    shutil.rmtree(extract_dir)

print('Extracting...')
os.makedirs(extract_dir, exist_ok=True)
with zipfile.ZipFile(zip_file, 'r') as zip_ref:
    zip_ref.extractall(extract_dir)

print('Extraction complete')

ZIP already exists
Extracting...
Extraction complete


In [15]:
# Locate a test folder if the ZIP contains one.
# If there is no official test folder, create /content/test_images from sample training images to verify inference.

image_exts = ('.jpg', '.jpeg', '.png')

def count_images(folder):
    total = 0
    for root, dirs, files in os.walk(folder):
        total += sum(f.lower().endswith(image_exts) for f in files)
    return total

def find_folder_by_name(base, names):
    for root, dirs, files in os.walk(base):
        # ignore macOS metadata folders
        if '__MACOSX' in root:
            continue
        for d in dirs:
            if d.lower() in names:
                path = os.path.join(root, d)
                if count_images(path) > 0:
                    return path
    return None

test_dir = find_folder_by_name(extract_dir, {'characters_test', 'test', 'test_images'})
train_dir = find_folder_by_name(extract_dir, {'characters_train'})

if test_dir is not None:
    data_dir = test_dir
    print('Found test folder:', data_dir)
else:
    if train_dir is None:
        raise FileNotFoundError('Could not find characters_test/test_images or characters_train in the ZIP.')

    print('No official test folder found. Creating sample /content/test_images from training images.')
    data_dir = '/content/test_images'
    if os.path.exists(data_dir):
        shutil.rmtree(data_dir)
    os.makedirs(data_dir, exist_ok=True)

    copied = 0
    max_images = 50
    for root, dirs, files in os.walk(train_dir):
        if '__MACOSX' in root:
            continue
        for f in files:
            if f.lower().endswith(image_exts):
                src = os.path.join(root, f)
                # prefix parent folder to avoid duplicate filenames
                class_name = os.path.basename(root)
                dst = os.path.join(data_dir, f'{class_name}_{f}')
                shutil.copy(src, dst)
                copied += 1
                if copied >= max_images:
                    break
        if copied >= max_images:
            break
    print(f'Created sample test folder: {data_dir} with {copied} images')

print('Final data_dir:', data_dir)
print('Number of images:', count_images(data_dir))

No official test folder found. Creating sample /content/test_images from training images.
Created sample test folder: /content/test_images with 50 images
Final data_dir: /content/test_images
Number of images: 50


In [16]:
# Model architecture MUST match train.ipynb

class SimpleCNN(nn.Module):
    def __init__(self, num_classes):
        super(SimpleCNN, self).__init__()

        self.features = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),

            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),

            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),

            nn.Conv2d(128, 256, kernel_size=3, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(),
            nn.MaxPool2d(2, 2)
        )

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(256 * 4 * 4, 512),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(512, num_classes)
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x

print("Model class ready")

Model class ready


In [17]:
def infer(data_dir, model_path):
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print('Using device:', device)

    if not os.path.exists(model_path):
        raise FileNotFoundError(f'Model file not found: {model_path}. Upload model.pth to Colab.')
    if not os.path.exists('classes.json'):
        raise FileNotFoundError('classes.json not found. Upload classes.json to Colab.')

    with open('classes.json', 'r') as f:
        classes = json.load(f)

    model = SimpleCNN(num_classes=len(classes))
    model.load_state_dict(torch.load(model_path, map_location=device))
    model.to(device)
    model.eval()

    transform = transforms.Compose([
        transforms.Resize((64, 64)),
        transforms.ToTensor()
    ])

    results = {}
    image_files = []
    for root, dirs, files in os.walk(data_dir):
        if '__MACOSX' in root:
            continue
        for file in files:
            if file.lower().endswith(('.jpg', '.jpeg', '.png')):
                image_files.append(os.path.join(root, file))

    if len(image_files) == 0:
        raise ValueError(f'No images found in {data_dir}')

    for path in sorted(image_files):
        filename = os.path.basename(path)
        image = Image.open(path).convert('RGB')
        image = transform(image).unsqueeze(0).to(device)

        with torch.no_grad():
            outputs = model(image)
            predicted = torch.argmax(outputs, dim=1).item()

        results[filename] = classes[predicted]

    with open('results.json', 'w') as f:
        json.dump(results, f, indent=4)

    print(f'results.json created with {len(results)} predictions')
    return results

print('infer() ready')

infer() ready


In [18]:
# Upload model.pth and classes.json to Colab before running this cell.
model_path = 'model.pth'

results = infer(data_dir, model_path)

# Show first 10 predictions
list(results.items())[:10]

Using device: cpu
results.json created with 50 predictions


[('abraham_grampa_simpson_pic_0004.jpg', 'homer_simpson'),
 ('abraham_grampa_simpson_pic_0008.jpg', 'homer_simpson'),
 ('abraham_grampa_simpson_pic_0025.jpg', 'homer_simpson'),
 ('abraham_grampa_simpson_pic_0064.jpg', 'homer_simpson'),
 ('abraham_grampa_simpson_pic_0077.jpg', 'homer_simpson'),
 ('abraham_grampa_simpson_pic_0079.jpg', 'homer_simpson'),
 ('abraham_grampa_simpson_pic_0082.jpg', 'homer_simpson'),
 ('abraham_grampa_simpson_pic_0087.jpg', 'homer_simpson'),
 ('abraham_grampa_simpson_pic_0088.jpg', 'homer_simpson'),
 ('abraham_grampa_simpson_pic_0089.jpg', 'homer_simpson')]